# Integrales-03 — La bulle et la première divergence

Le carnet 01 calculait le triangle à une boucle : en $d = 2$, tout y restait fini.
Le carnet 02 poussait au sunrise et voyait les polylogarithmes sortir. Celui-ci
descend d'un étage — **deux propagateurs, la bulle** — et monte à $d = 4$ :

$$B(\Lambda) = \int_{|k| < \Lambda} \frac{d^4 k}{k^2 (k-p)^2}, \qquad p^2 = 1 .$$

C'est le plus petit graphe dont l'intégrale **diverge** : l'intégrande décroît
comme $1/k^4$ en $|k| \to \infty$, exactement la dimension du volume —
l'intégrale croît comme $\ln \Lambda^2$. Le programme du carnet :

1. **la mesure de la sphère $S^3$** — vérifiée numériquement avant tout usage
   (une première écriture de ce carnet utilisait $2\pi\sin^2\psi\, d\psi$ et
   rendait la moitié de l'intégrale : le contrôle importe) ;
2. **une forme fermée** par paramètre de Feynman, avec son domaine de validité
   exhibé (la boule dure ne commute pas au décalage — l'écart est mesuré, pas
   masqué) ;
3. **la divergence mesurée** : coefficient de $\ln \Lambda^2$ et constante
   finie, chacune ancrée dans une *planted truth* indépendante ;
4. **la régularisation soustractive** : un paramètre $\mu$, une limite finie.

Comme dans les carnets 01 et 02 : outil réel (`mpmath`, 30 chiffres), chaque
affirmation quantitative vit dans une sortie commise, les refus sont documentés.

## 1. La mesure de $S^3$

En quatre dimensions, $d^4 k = k^3\,dk\,d\Omega_3$ et la sphère unité se
paramètre par l'angle $\psi$ entre $k$ et $p$ :

$$d\Omega_3 = \sin^2\psi\, d\psi\, d\Omega_2, \qquad \int d\Omega_2 = 4\pi
\quad\Longrightarrow\quad \int_{S^3} d\Omega_3 = 4\pi \int_0^\pi \sin^2\psi\, d\psi = 2\pi^2 .$$

La cellule suivante vérifie l'aire par quadrature — c'est la *planted truth* de
la mesure : toute la suite en dépend.

In [1]:
from mpmath import mp, mpf, quad, pi, log, sqrt, sin, cos
mp.dps = 30
p2 = mpf(1)  # p^2 = 1 : l'echelle

# aire de S^3 par quadrature (planted truth de la mesure)
aire = quad(lambda psi: 4*pi*sin(psi)**2, [0, pi])
print("aire S^3 mesuree =", aire)
print("2 pi^2           =", 2*pi**2)
print("ecart relatif    =", abs((aire - 2*pi**2) / (2*pi**2)))

aire S^3 mesuree = 19.7392088021787172376689819998
2 pi^2           = 19.7392088021787172376689819998
ecart relatif    = 0.0


## 2. La forme fermée : paramètre de Feynman et complétion du carré

L'identité $\frac{1}{AB} = \int_0^1 \frac{dx}{(xA + (1-x)B)^2}$ avec
$A = k^2$, $B = (k-p)^2$ donne, après complétion du carré,

$$\frac{1}{k^2 (k-p)^2} = \int_0^1 \frac{dx}{\big((k - (1-x)p)^2 + \Delta\big)^2},
\qquad \Delta = x(1-x)\, p^2 .$$

**Avertissement honnête** : la forme fermée ci-dessous échange
$\int_{|k|<\Lambda}$ contre $\int_{|q|<\Lambda}$ après le décalage
$q = k - (1-x)p$. Ce geste est exact en régularisation dimensionnelle (où la
translation est une invariance), mais une **boule dure ne commute pas au
décalage** : l'écart est d'ordre $p/\Lambda$, mesuré en section 3 — pas masqué.
L'intégrale radiale, elle, est exacte :

$$\int_0^\Lambda \frac{k^3\,dk}{(k^2+\Delta)^2} = \frac{1}{2}\left[
\ln\frac{\Lambda^2+\Delta}{\Delta} + \frac{\Delta}{\Lambda^2+\Delta} - 1\right].$$

In [2]:
def B_ferme(Lam, p2=p2):
    """Forme fermee (parametre de Feynman + radial exact). Valide a O(p/Lam) pres
    (boule dure vs carre complete) : l'ecart est mesure en section 3."""
    f = lambda x: (log((Lam**2 + x*(1-x)*p2) / (x*(1-x)*p2))
                   + x*(1-x)*p2/(Lam**2 + x*(1-x)*p2) - 1)
    return pi**2 * quad(f, [0, 1])   # 2 pi^2 * 1/2 = pi^2 devant le crochet

for Lam in (mpf(2), mpf(8)):
    print(f"B_ferme(Lam={int(Lam)}) =", B_ferme(Lam))

B_ferme(Lam=2) = 24.34480199630533702092892284
B_ferme(Lam=8) = 50.9674192526556407971513075656


## 3. La voie directe : quadrature sur la boule, et le facteur 2 attrapé en direct

La voie indépendante n'utilise **aucun** paramètre de Feynman : elle intègre
l'intégrande d'origine sur $\psi \in [0, \pi]$ puis $k \in [0, \Lambda]$.
La première version de ce carnet écrivait la mesure angulaire
$2\pi\sin^2\psi$ — soit la moitié de $4\pi\sin^2\psi$ — et rendait
exactement la moitié de l'intégrale. Le contrôle d'aire de la section 1 est ce
qui tranche ; l'écart restant entre les deux voies, lui, n'est **pas** une
erreur : c'est le $O(p/\Lambda)$ du décalage de la section 2, et il décroît
quand $\Lambda$ croît.

In [3]:
def B_direct(Lam, p2=p2):
    """Quadrature 2D directe sur la boule (aucun parametre de Feynman)."""
    p = sqrt(p2)
    def radial(k):
        g = lambda psi: (4*pi*sin(psi)**2 /
                         (k**2 * (k**2 + p2 - 2*k*p*cos(psi))))
        return k**3 * quad(g, [0, pi])
    return quad(radial, [0, mpf('0.5')*p, 2*p, Lam])

for Lam in (mpf(2), mpf(8)):
    bf, bd = B_ferme(Lam), B_direct(Lam)
    print(f"Lam={int(Lam):2d} : ferme = {bf}")
    print(f"          direct = {bd}   ecart rel = {abs((bf-bd)/bf)}")

Lam= 2 : ferme = 24.34480199630533702092892284
          direct = 23.5519890022914087237865738071   ecart rel = 0.032566007073470908403891263134


Lam= 8 : ferme = 50.9674192526556407971513075656
          direct = 50.9163428577198749031764904229   ecart rel = 0.00100213814402824751687136357404


## 4. La divergence : $B(\Lambda) - \pi^2 \ln \Lambda^2$ tend vers une constante

Le comptage de puissances prédit $B(\Lambda) = \pi^2\big(\ln\Lambda^2 +
c\big) + O(1/\Lambda^2)$. Le tableau mesure la soustraction — la constante
émerge chiffre après chiffre :

In [4]:
print(f"{'Lam':>8} {'B(Lam) - pi^2 ln(Lam^2)':>32} {'residu/precedent':>18}")
prev = None
for e in (2, 4, 6, 8):
    Lam = mpf(10)**e
    val = B_ferme(Lam) - pi**2 * log(Lam**2)
    rat = "  --" if prev is None else mp.nstr(val/prev, 8)
    print(f"Lam=1e{e}  B - pi^2 ln(Lam^2) = {mp.nstr(val, 22)}  ratio = {rat}")
    prev = val

     Lam          B(Lam) - pi^2 ln(Lam^2)   residu/precedent
Lam=1e2  B - pi^2 ln(Lam^2) = 9.869933382968020057851  ratio =   --
Lam=1e4  B - pi^2 ln(Lam^2) = 9.869604433988039906451  ratio = 0.99996667
Lam=1e6  B - pi^2 ln(Lam^2) = 9.869604401092648486968  ratio = 1.0
Lam=1e8  B - pi^2 ln(Lam^2) = 9.869604401089358947821  ratio = 1.0


## 5. Planted truth n° 1 — le coefficient de $\ln \Lambda^2$ est $\pi^2$

La différence $B(10\Lambda) - B(\Lambda)$ isole le coefficient sans aucune
connaissance de la constante : divisée par $\ln 100$, elle doit rendre $\pi^2$
exactement.

In [5]:
L = mpf(10)**6
coef = (B_ferme(10*L) - B_ferme(L)) / log(mpf(100))
print("coefficient mesure =", coef)
print("pi^2               =", pi**2)
print("ecart relatif      =", abs((coef - pi**2) / pi**2))

coefficient mesure = 9.86960440108865137690404751611
pi^2               = 9.86960440108935861883449099988
ecart relatif      = 7.16585895140256955602111645009e-14


## 6. Planted truth n° 2 — la constante finie est $\pi^2$, ancrée dans l'intégrale $x$

Le développement de la forme fermée donne
$B(\Lambda) = \pi^2\big(\ln\Lambda^2 + \int_0^1 \ln\frac{1}{x(1-x)}\,dx
- 1\big) + o(1)$. L'intégrale $x$ se calcule **à la main** :
$\int_0^1 \ln x\,dx = \int_0^1 \ln(1-x)\,dx = -1$, donc la constante vaut
$\pi^2(2 - 1) = \pi^2$. Les deux cellules mesurent l'ancre et la limite — par
deux routes indépendantes (grande $\Lambda$ ; voie directe tronquée, dont le
résidu prédit $\sim\bar\Delta/\Lambda^2$).

In [6]:
# ancre : l'integrale x, calculee par quadrature (et verifiee a la main : -1 + -1 = -2)
int_x = quad(lambda x: log(1/(x*(1-x))), [0, 1])
print("int_0^1 ln(1/(x(1-x))) dx =", int_x, "  (a la main : 2)")
F_theo = pi**2 * (int_x - 1)

# limite grande Lambda
F_num = B_ferme(mpf(10)**12) - pi**2 * log(mpf(10)**24)
print("F mesure (Lambda=1e12)     =", F_num)
print("F theo  pi^2 (2 - 1)       =", F_theo)
print("ecart relatif              =", abs((F_num - F_theo) / F_theo))

int_0^1 ln(1/(x(1-x))) dx = 2.0   (a la main : 2)
F mesure (Lambda=1e12)     = 9.86960440108935861883449428972
F theo  pi^2 (2 - 1)       = 9.86960440108935861883449099988
ecart relatif              = 3.33330510716480601405036945999e-25


In [7]:
# voie directe tronquee : residu predit en Delta/Lambda^2, mesure ~2e-5 a Lambda=64
Fd = B_direct(mpf(64)) - pi**2 * log(mpf(64)**2)
print("F voie directe (Lambda=64) =", Fd)
print("ecart vs F_theo            =", abs((Fd - F_theo) / F_theo))

F voie directe (Lambda=64) = 9.86981207457717563409161549917
ecart vs F_theo            = 0.0000210417235967525987332626790563


## 7. La régularisation soustractive : un paramètre, une échelle, une limite finie

Le geste du carnet 01 face à ses limites, transposé : on **soustrait** le
divergent à une échelle $\mu$,

$$B_R(\mu^2) = \lim_{\Lambda \to \infty}\; B(\Lambda) - \pi^2 \ln\frac{\Lambda^2}{\mu^2} = \pi^2\big(1 + \ln\mu^2\big) \quad (p^2 = 1).$$

La limite existe, est finie, et **dépend de l'échelle** : c'est la première
apparition, dans la série, d'une quantité qui « court » —
$\partial B_R / \partial \ln \mu^2 = \pi^2$, mesuré dans la sortie.

In [8]:
def B_R(mu2, Lam=mpf(10)**12, p2=p2):
    return B_ferme(Lam, p2) - pi**2 * log(Lam**2 / mu2)

for mu2 in (mpf(1), mpf(4), mpf(16)):
    val = B_R(mu2)
    print(f"mu^2 = {int(mu2):2d} : B_R = {val}   pi^2 (1 + ln mu^2) = {pi**2*(1+log(mu2))}")

# dependance d'echelle mesuree par differences finies
d = (B_R(mpf(4)) - B_R(mpf(1))) / log(mpf(4))
print("dB_R/d ln(mu^2) mesure =", d, "  pi^2 =", pi**2)

mu^2 =  1 : B_R = 9.86960440108935861883449428972   pi^2 (1 + ln mu^2) = 9.86960440108935861883449099988
mu^2 =  4 : B_R = 23.5517813288035917085294525976   pi^2 (1 + ln mu^2) = 23.5517813288035917085294493078
mu^2 = 16 : B_R = 37.2339582565178247982244109056   pi^2 (1 + ln mu^2) = 37.2339582565178247982244076157
dB_R/d ln(mu^2) mesure = 9.86960440108935861883449099986   pi^2 = 9.86960440108935861883449099988


## 8. Refus documentés (reconnaissance de constantes, anneau déclaré avant recherche)

Avant toute recherche de relation, l'anneau des constantes est déclaré :
$\{\pi, \ln 2, \gamma_E, \ln 4\pi\}$.

- **$B(\Lambda)/\ln\Lambda^2 \to 1$** — mesuré ; chercher une autre
  constante multiplicative serait fabriquer un résultat.
- **La partie finie $F = \pi^2$ ne contient ni $\gamma_E$ ni $\ln 4\pi$** —
  ces constantes appartiennent au schéma de régularisation dimensionnelle,
  citée ('t Hooft–Veltman 1979, déjà aux Sources de la série) et non reproduite
  ici : la boule dure les ignore, et c'est une **différence de schéma**, pas
  une contradiction.
- **Le résidu $R(\Lambda) = B(\Lambda) - \pi^2(\ln\Lambda^2+1)$ décroît en
  $1/\Lambda^2$**, comme le prédit le terme $\bar\Delta/\Lambda^2$ de la
  forme fermée — vérifié au ratio ci-dessous plutôt qu'affirmé.

In [9]:
def residu(Lam, p2=p2):
    return B_ferme(Lam, p2) - pi**2*(log(Lam**2) + 1)

r2 = residu(mpf(10)**2)
r4 = residu(mpf(10)**4)
print("R(1e2) =", r2)
print("R(1e4) =", r4)
print("ratio R(1e2)/R(1e4) =", r2/r4, " (decroissance en 1/Lambda^2 : attendu 1e4)")

R(1e2) = 0.000328981878661439016794383968072
R(1e4) = 0.0000000328986812876165068180021336187
ratio R(1e2)/R(1e4) = 9999.85001785685833284495861  (decroissance en 1/Lambda^2 : attendu 1e4)


## 9. Exercices

Trois exercices, à compléter — le carnet s'exécute de bout en bout sans eux
(règle C.1).

In [10]:
# Exercice 1 -- La bulle ne connait p^2 que par le logarithme.
# Refaire le tableau de la section 4 avec p2 = 4, puis verifier que
# B(Lam; p2=4) - pi^2*(ln(Lam^2) - ln(4) + 1) tend vers 0.
# Indice : la prediction vient de la forme fermee (remplacer Delta -> 4 x(1-x)).
# Etape 1 : poser p2_exo = mpf(4) et reevaluer B_ferme(1e8, p2_exo) - pi**2*(log(mpf(10)**16) - log(mpf(4)) + 1)
resultat_exo1 = None  # TODO etudiant
print("Exercice a completer : dependance d'echelle de la bulle")

Exercice a completer : dependance d'echelle de la bulle


In [11]:
# Exercice 2 -- La meme bulle en d = 3.
# En trois dimensions la mesure est S^2 (aire 4*pi) et l'integrande decroit
# comme 1/k^4 sur un volume en k^2 dk : le comptage de puissances predit la
# convergence. Mesurer B3(Lam) a Lam = 10, 100 et verifier que la suite se
# stabilise (contrairement au tableau de la section 4).
# Etape 1 : ecrire B3_direct(Lam) sur le modele de B_direct, avec la mesure
#           2*pi*sin(psi) dpsi et k**2 dk (volume S^2).
resultat_exo2 = None  # TODO etudiant
print("Exercice a completer : bulle en d=3, convergence")

Exercice a completer : bulle en d=3, convergence


In [12]:
# Exercice 3 -- Verifier la forme radiale fermee par differences finies.
# La section 2 utilise integral_0^Lam k^3 dk/(k^2+Delta)^2 = 1/2[ln((Lam^2+Delta)/Delta)
# + Delta/(Lam^2+Delta) - 1]. Verifier numeriquement (quad direct vs formule)
# pour Delta = 0.1 et Lam = 3, a 25 chiffres.
# Etape 1 : comparer quad(lambda k: k**3/(k**2+mpf('0.1'))**2, [0, 3]) a la formule.
resultat_exo3 = None  # TODO etudiant
print("Exercice a completer : forme radiale fermee")

Exercice a completer : forme radiale fermee


## 10. Ce que ce carnet établit

- La **première divergence** de la série : la bulle en $d = 4$ croît comme
  $\pi^2 \ln \Lambda^2$, coefficient mesuré à $7 \times 10^{-14}$ (planted
  truth n° 1).
- La **constante finie $F = \pi^2$**, ancrée dans une intégrale calculable à
  la main ($\int_0^1 \ln 1/x(1-x)\,dx = 2$), mesurée à $3 \times 10^{-25}$
  par la forme fermée et à $2 \times 10^{-5}$ par la voie directe tronquée —
  résidu prédit et observé en $\bar\Delta/\Lambda^2$.
- La **régularisation soustractive** : $B_R(\mu^2) = \pi^2(1 + \ln\mu^2)$,
  limite finie exhibée, dépendance d'échelle mesurée — le premier « running »
  de la série.
- Deux **leçons de méthode** exposées, pas gommées : la mesure $S^3$ vérifiée
  avant usage (une première écriture rendait la moitié), et l'écart
  $O(p/\Lambda)$ de la boule dure face au décalage, mesuré en section 3.

L'étage suivant du classement BootLoops (polylogarithmes → **elliptique** →
K3 → Calabi-Yau) reste ouvert : la structure de double période entrevue au
carnet 02 n'est pas développée ici. Le candidat 04 — s'il vit — la reprendra
depuis la courbe elliptique du sunrise.